# 03 MSA: crossed Gage R&R for the crimp height gauge

Simulated study (`python -m src.simulate.gage`): 10 parts from the crimp process, 3 operators,
3 trials each. Crossed ANOVA with the part x operator interaction, interaction pooled when
p >= 0.05, then variance components, %Study Var, %Tolerance (tolerance 0.10 mm) and ndc.

Acceptance: %GRR < 10% accept, 10-30% conditional, > 30% reject; ndc >= 5.
The truth file is opened only in the last section.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # make `src` importable when run from notebooks/

In [ ]:
import json

import pandas as pd

from src.msa import anova_crossed, decision, gage_evaluation, variance_components
from src.plots import grr_by_part_chart, grr_study_var_chart

FIGURES = ROOT / "reports" / "figures"
TABLES = ROOT / "reports" / "tables"
TOLERANCE = 0.10

study = pd.read_csv(ROOT / "data" / "simulated" / "gage_study.csv")
print(
    len(study),
    "readings;",
    study["part"].nunique(),
    "parts,",
    study["operator"].nunique(),
    "operators,",
    study["trial"].nunique(),
    "trials",
)

## 1. ANOVA

In [ ]:
anova = anova_crossed(study, "part", "operator", "crimp_height_mm")
anova.to_csv(TABLES / "msa_anova.csv", index=False)
anova.round(8)

## 2. Variance components and gauge evaluation

In [ ]:
vc = variance_components(anova, parts=10, operators=3, trials=3, alpha=0.05)
print("interaction p =", round(vc["interaction_p"], 3), "-> removed:", vc["interaction_removed"])
evaluation = gage_evaluation(vc, tolerance=TOLERANCE)
evaluation.round(6).to_csv(TABLES / "msa_gage_evaluation.csv", index=False)
ndc = evaluation.attrs["ndc"]
pct_grr = evaluation.set_index("source").loc["Total Gage R&R", "pct_study_var"]
print(f"%GRR (study variation) = {pct_grr:.1f}%, ndc = {ndc}, decision: {decision(pct_grr, ndc)}")
evaluation.round(4)

In [ ]:
grr_study_var_chart(
    evaluation,
    "Crimp height gauge: share of study variation",
    FIGURES / "msa_study_var.png",
    eyebrow="MSA · Crossed Gage R&R",
    source="Simulated data (seeded), not plant measurements",
)
grr_by_part_chart(
    study,
    "Gage R&R readings by part and operator",
    FIGURES / "msa_by_part.png",
    eyebrow="MSA · Crossed Gage R&R",
    source="Simulated data (seeded), not plant measurements",
)

![Study variation](../reports/figures/msa_study_var.png)

![Readings by part](../reports/figures/msa_by_part.png)

## 3. Operator means

Reproducibility comes from differences between operators; their mean readings over the same
parts show which operator reads high or low.

In [ ]:
op_means = study.groupby("operator")["crimp_height_mm"].mean()
(op_means - op_means.mean()).rename("deviation_from_overall_mm").round(5)

## 4. Scoring against the simulated truth

In [ ]:
truth = json.loads((ROOT / "data" / "simulated" / "gage_truth.json").read_text())
pop = truth["population"]
ev = evaluation.set_index("source")
comparison = pd.DataFrame(
    {
        "estimated": [
            ev.loc["Total Gage R&R", "std_dev"],
            pct_grr,
            ev.loc["Total Gage R&R", "pct_tolerance"],
            ndc,
        ],
        "true (population)": [
            pop["gage_rr_sd"],
            pop["pct_study_var"],
            pop["pct_tolerance"],
            pop["ndc"],
        ],
    },
    index=["Gage R&R sd (mm)", "%Study Var", "%Tolerance", "ndc"],
)
comparison.round(4).to_csv(TABLES / "msa_vs_truth.csv")
print("true operator biases (mm):", truth["operator_bias_mm"])
comparison.round(4)

## 5. How stable is this estimate?

Rerun the same study design 300 times with new random parts, operator biases and noise
(different seeds, same settings) and look at the spread of the estimates.

In [ ]:
from src.simulate.gage import GageConfig, simulate_study

reps = []
for seed in range(300):
    df, _ = simulate_study(GageConfig(seed=seed))
    v = variance_components(anova_crossed(df, "part", "operator", "crimp_height_mm"), 10, 3, 3)
    e = gage_evaluation(v, tolerance=TOLERANCE)
    g = e.set_index("source").loc["Total Gage R&R", "pct_study_var"]
    reps.append(
        {
            "operator_var": v["operator"],
            "repeatability_var": v["repeatability"],
            "pct_grr": g,
            "ndc": e.attrs["ndc"],
            "decision": decision(g, e.attrs["ndc"]),
        }
    )
reps = pd.DataFrame(reps)
summary = pd.Series(
    {
        "mean operator variance (true 1.44e-06)": reps["operator_var"].mean(),
        "share of studies with operator variance 0": (reps["operator_var"] == 0).mean(),
        "mean repeatability variance (true 4.08e-06 incl. rounding)": reps[
            "repeatability_var"
        ].mean(),
        "%GRR 5th percentile": reps["pct_grr"].quantile(0.05),
        "%GRR median": reps["pct_grr"].median(),
        "%GRR 95th percentile": reps["pct_grr"].quantile(0.95),
    }
)
summary.to_csv(TABLES / "msa_repeated_studies.csv", header=["value"])
print(reps["decision"].value_counts(normalize=True).round(3).to_string())
summary.map(lambda v: f"{v:.3g}")